# ClipCap dataset inference

Notebook này chỉ thực hiện **inference**, không tính metric. Có thể chọn `val` hoặc `test`, chạy một hay cả năm Mapper, và lưu caption dưới dạng `predictions.jsonl` để notebook `clipcap_evaluation.ipynb` sử dụng. Inference manifest có thể chỉ là danh sách image ID; ground-truth caption không cần xuất hiện trong luồng này.

Luồng xử lý: ảnh hoặc CLIP feature cache -> Mapper `final.pt` -> GPT-2 beam search -> 5 candidate -> CLIP reranking -> 1 caption cuối. Không sử dụng hard prompt.

## 1. Cấu hình run

Khi chạy độc lập, chỉnh các biến mặc định trong cell dưới. Notebook end-to-end Colab có thể truyền cùng cấu hình qua biến môi trường. Test bị khóa mặc định để tránh dùng test trong quá trình tuning.

In [ ]:
from __future__ import annotations

import hashlib
import json
import os
import re
import subprocess
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display


def find_project_root(start: Path) -> Path:
    resolved = start.expanduser().resolve()
    for candidate in (resolved, *resolved.parents):
        if (candidate / 'src' / 'config' / 'common_config.py').is_file():
            return candidate
    raise FileNotFoundError(
        'Không tìm thấy project root. Hãy mở notebook trong repository hoặc '
        'đặt biến môi trường ZFS_CLIP_PROJECT_ROOT.'
    )


def env_bool(name: str, default: bool) -> bool:
    value = os.environ.get(name)
    if value is None:
        return default
    normalized = value.strip().lower()
    if normalized in {'1', 'true', 'yes', 'on'}:
        return True
    if normalized in {'0', 'false', 'no', 'off'}:
        return False
    raise ValueError(f'{name} phải là biến boolean')


def env_path(name: str, default: Path | None = None) -> Path | None:
    value = os.environ.get(name)
    return Path(value).expanduser() if value else default


project_override = env_path('ZFS_CLIP_PROJECT_ROOT')
PROJECT_ROOT = find_project_root(project_override or Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config.clipcap_config import (
    CLIPCAP_DEFAULT_INFERENCE_CONFIG,
    CLIPCAP_OUTPUT_ROOT,
    CLIPCAP_TRAIN_SEED,
    CLIPCAP_TRAIN_SUBSETS,
    CLIP_MODEL_NAME,
    ClipCapInferenceConfig,
)
from src.config.common_config import RAW_IMAGES_DIR, SPLIT_DIR

SPLIT_NAME = os.environ.get('ZFS_CLIP_SPLIT_NAME', 'val')
ALLOW_TEST_INFERENCE = env_bool('ZFS_CLIP_ALLOW_TEST', False)
RUN_TAG = os.environ.get('ZFS_CLIP_RUN_TAG', 'baseline_v1')
RUN_INFERENCE = env_bool('ZFS_CLIP_RUN_INFERENCE', True)
SEED = int(os.environ.get('ZFS_CLIP_SEED', str(CLIPCAP_TRAIN_SEED)))

subset_text = os.environ.get(
    'ZFS_CLIP_SUBSETS',
    ','.join(CLIPCAP_TRAIN_SUBSETS),
)
SUBSETS_TO_RUN = tuple(item.strip() for item in subset_text.split(',') if item.strip())

INFERENCE_CONFIG = ClipCapInferenceConfig(
    image_batch_size=int(os.environ.get(
        'ZFS_CLIP_IMAGE_BATCH_SIZE',
        str(CLIPCAP_DEFAULT_INFERENCE_CONFIG.image_batch_size),
    )),
    max_new_tokens=int(os.environ.get(
        'ZFS_CLIP_MAX_NEW_TOKENS',
        str(CLIPCAP_DEFAULT_INFERENCE_CONFIG.max_new_tokens),
    )),
    num_beams=int(os.environ.get(
        'ZFS_CLIP_NUM_BEAMS',
        str(CLIPCAP_DEFAULT_INFERENCE_CONFIG.num_beams),
    )),
    num_return_sequences=int(os.environ.get(
        'ZFS_CLIP_NUM_RETURN_SEQUENCES',
        str(CLIPCAP_DEFAULT_INFERENCE_CONFIG.num_return_sequences),
    )),
    length_penalty=float(os.environ.get(
        'ZFS_CLIP_LENGTH_PENALTY',
        str(CLIPCAP_DEFAULT_INFERENCE_CONFIG.length_penalty),
    )),
    early_stopping=env_bool(
        'ZFS_CLIP_EARLY_STOPPING',
        CLIPCAP_DEFAULT_INFERENCE_CONFIG.early_stopping,
    ),
)
MAX_NEW_TOKENS = INFERENCE_CONFIG.max_new_tokens
NUM_BEAMS = INFERENCE_CONFIG.num_beams
NUM_RETURN_SEQUENCES = INFERENCE_CONFIG.num_return_sequences
LENGTH_PENALTY = INFERENCE_CONFIG.length_penalty
EARLY_STOPPING = INFERENCE_CONFIG.early_stopping
IMAGE_BATCH_SIZE = INFERENCE_CONFIG.image_batch_size
DEVICE = os.environ.get('ZFS_CLIP_DEVICE', 'auto')

MANIFEST_PATH = env_path(
    'ZFS_CLIP_INFERENCE_MANIFEST_PATH',
    env_path(
        'ZFS_CLIP_MANIFEST_PATH',
        SPLIT_DIR / f'{SPLIT_NAME}.json',
    ),
)
CHECKPOINT_ROOT = env_path('ZFS_CLIP_CHECKPOINT_ROOT', CLIPCAP_OUTPUT_ROOT)
OUTPUT_BASE = env_path(
    'ZFS_CLIP_INFERENCE_OUTPUT_BASE',
    CLIPCAP_OUTPUT_ROOT / 'evaluation',
)
INFERENCE_OUTPUT_ROOT = OUTPUT_BASE / SPLIT_NAME / RUN_TAG

precomputed_cache = PROJECT_ROOT / 'data' / 'flickr8k' / 'features' / 'clip_features.pt'
default_cache = (
    precomputed_cache
    if precomputed_cache.is_file()
    else CLIPCAP_OUTPUT_ROOT / 'feature_cache' / f'{SPLIT_NAME}_clip_features.pt'
)
FEATURE_CACHE_PATH = env_path('ZFS_CLIP_FEATURE_CACHE', default_cache)
IMAGE_DIR = env_path('ZFS_CLIP_IMAGE_DIR')
if IMAGE_DIR is None and Path(RAW_IMAGES_DIR).is_dir():
    IMAGE_DIR = Path(RAW_IMAGES_DIR)

if SPLIT_NAME not in {'val', 'test'}:
    raise ValueError("SPLIT_NAME phải là 'val' hoặc 'test'")
if SPLIT_NAME == 'test' and not ALLOW_TEST_INFERENCE:
    raise RuntimeError(
        'Test inference đang bị khóa. Chỉ bật sau khi đã chốt cấu hình bằng validation.'
    )
if not RUN_TAG or not re.fullmatch(r'[A-Za-z0-9._-]+', RUN_TAG):
    raise ValueError('RUN_TAG chứa ký tự không hợp lệ')
if not SUBSETS_TO_RUN:
    raise ValueError('SUBSETS_TO_RUN không được rỗng')
unknown_subsets = set(SUBSETS_TO_RUN) - set(CLIPCAP_TRAIN_SUBSETS)
if unknown_subsets:
    raise ValueError(f'Unknown subsets: {sorted(unknown_subsets)}')
if not MANIFEST_PATH.is_file():
    raise FileNotFoundError(f'Không tìm thấy manifest: {MANIFEST_PATH}')
if not FEATURE_CACHE_PATH.is_file() and (IMAGE_DIR is None or not IMAGE_DIR.is_dir()):
    raise FileNotFoundError(
        'Feature cache chưa tồn tại; cần đặt ZFS_CLIP_IMAGE_DIR để tạo cache.'
    )

print(f'Project root: {PROJECT_ROOT}')
print(f'Split: {SPLIT_NAME} | Run tag: {RUN_TAG}')
print(f'Subsets: {list(SUBSETS_TO_RUN)}')
print(f'Manifest: {MANIFEST_PATH}')
print(f'Checkpoint root: {CHECKPOINT_ROOT}')
print(f'Feature cache: {FEATURE_CACHE_PATH}')
print(f'Output: {INFERENCE_OUTPUT_ROOT}')

## 2. Chạy inference

Cell này gọi pipeline `.py` để tránh sao chép logic model vào notebook. `final.pt`, no-prompt, beam search và CLIP reranking đều được kiểm tra bởi source module. Chạy lại cùng cấu hình sẽ resume từ các ảnh đã hoàn thành.

In [ ]:
for subset_name in SUBSETS_TO_RUN:
    checkpoint_dir = CHECKPOINT_ROOT / subset_name / f'seed_{SEED}'
    final_path = checkpoint_dir / 'final.pt'
    config_path = checkpoint_dir / 'config.json'
    if not final_path.is_file() or not config_path.is_file():
        raise FileNotFoundError(
            f'Thiếu final.pt hoặc config.json cho {subset_name}: {checkpoint_dir}'
        )

inference_command = [
    sys.executable,
    '-m',
    'src.clipcap.inference',
    '--manifest',
    str(MANIFEST_PATH),
    '--feature-cache',
    str(FEATURE_CACHE_PATH),
    '--checkpoint-root',
    str(CHECKPOINT_ROOT),
    '--subsets',
    *SUBSETS_TO_RUN,
    '--seed',
    str(SEED),
    '--output-dir',
    str(INFERENCE_OUTPUT_ROOT),
    '--device',
    DEVICE,
    '--image-batch-size',
    str(IMAGE_BATCH_SIZE),
    '--max-new-tokens',
    str(MAX_NEW_TOKENS),
    '--num-beams',
    str(NUM_BEAMS),
    '--num-return-sequences',
    str(NUM_RETURN_SEQUENCES),
    '--length-penalty',
    str(LENGTH_PENALTY),
]
if IMAGE_DIR is not None:
    inference_command.extend(['--image-dir', str(IMAGE_DIR)])
inference_command.append(
    '--early-stopping' if EARLY_STOPPING else '--no-early-stopping'
)

print('Inference command:')
print(subprocess.list2cmdline(inference_command))
if RUN_INFERENCE:
    subprocess.run(inference_command, cwd=PROJECT_ROOT, check=True)
else:
    print('RUN_INFERENCE=False: chỉ hiển thị lệnh, không thực thi.')

## 3. Xác minh output

Mỗi subset phải có đúng một prediction cho từng ảnh trong manifest, không trùng ID, dùng `final.pt`, và chứa đúng số candidate đã cấu hình. `run_config.json` được đối chiếu để ngăn trộn hai thí nghiệm.

In [ ]:
def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as file:
        for chunk in iter(lambda: file.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()


with MANIFEST_PATH.open('r', encoding='utf-8') as file:
    manifest = json.load(file)
if isinstance(manifest, dict):
    manifest_ids = list(manifest)
elif isinstance(manifest, list):
    manifest_ids = manifest
else:
    raise TypeError('Inference manifest phải là JSON object hoặc list')
if not manifest_ids or not all(
    isinstance(image_id, str) and image_id.strip() for image_id in manifest_ids
):
    raise ValueError('Inference manifest chứa image_id không hợp lệ')
if len(manifest_ids) != len(set(manifest_ids)):
    raise ValueError('Inference manifest chứa image_id trùng')
expected_ids = set(manifest_ids)

run_config_path = INFERENCE_OUTPUT_ROOT / 'run_config.json'
if not run_config_path.is_file():
    raise FileNotFoundError(f'Không tìm thấy run config: {run_config_path}')
with run_config_path.open('r', encoding='utf-8') as file:
    run_config = json.load(file)

expected_run_values = {
    'manifest_sha256': sha256_file(MANIFEST_PATH),
    'subsets': list(SUBSETS_TO_RUN),
    'seed': SEED,
    'checkpoint': 'final.pt',
    'training_policy': 'fixed_epoch',
    'prompt': None,
    'max_new_tokens': MAX_NEW_TOKENS,
    'num_beams': NUM_BEAMS,
    'num_return_sequences': NUM_RETURN_SEQUENCES,
    'length_penalty': LENGTH_PENALTY,
    'early_stopping': EARLY_STOPPING,
    'num_images': len(expected_ids),
    'clip_model': CLIP_MODEL_NAME,
}
for key, expected_value in expected_run_values.items():
    if run_config.get(key) != expected_value:
        raise ValueError(
            f'run_config mismatch tại {key}: '
            f'{run_config.get(key)!r} != {expected_value!r}'
        )

coverage_rows = []
sample_rows = []
for subset_name in SUBSETS_TO_RUN:
    prediction_path = (
        INFERENCE_OUTPUT_ROOT / subset_name / f'seed_{SEED}' / 'predictions.jsonl'
    )
    if not prediction_path.is_file():
        raise FileNotFoundError(f'Không tìm thấy prediction: {prediction_path}')
    records = []
    with prediction_path.open('r', encoding='utf-8') as file:
        for line_number, line in enumerate(file, start=1):
            if not line.strip():
                continue
            try:
                record = json.loads(line)
            except json.JSONDecodeError as error:
                raise ValueError(
                    f'JSONL lỗi tại {prediction_path}:{line_number}'
                ) from error
            records.append(record)
    prediction_ids = [record.get('image_id') for record in records]
    if len(prediction_ids) != len(set(prediction_ids)):
        raise ValueError(f'{subset_name} chứa image_id trùng')
    missing = expected_ids - set(prediction_ids)
    extra = set(prediction_ids) - expected_ids
    if missing or extra:
        raise ValueError(
            f'{subset_name}: missing={len(missing)}, extra={len(extra)}'
        )
    for record in records:
        if record.get('subset_name') != subset_name:
            raise ValueError(f'{subset_name}: subset metadata không khớp')
        if record.get('checkpoint') != 'final.pt':
            raise ValueError(f'{subset_name}: prediction không dùng final.pt')
        if len(record.get('candidates', [])) != NUM_RETURN_SEQUENCES:
            raise ValueError(f'{subset_name}: số candidate không đúng')
    coverage_rows.append({
        'subset': subset_name,
        'predictions': len(records),
        'expected': len(expected_ids),
        'complete': len(records) == len(expected_ids),
    })
    for record in records[:3]:
        sample_rows.append({
            'subset': subset_name,
            'image_id': record['image_id'],
            'caption': record['caption'],
            'selected_beam_rank': record['selected_beam_rank'],
        })

display(pd.DataFrame(coverage_rows))
display(pd.DataFrame(sample_rows))
print('Inference output đã sẵn sàng cho clipcap_evaluation.ipynb')

## 4. Quy tắc sử dụng

- Mỗi cấu hình decoding phải dùng một `RUN_TAG` riêng.
- Tuning chỉ thực hiện trên validation.
- Sau khi chốt cấu hình, đổi sang test, bật khóa test một lần và không tiếp tục tuning bằng kết quả test.
- Notebook này không tính metric; output của nó là input cho `clipcap_evaluation.ipynb`.